## Imports

In [12]:
import polars as pl
from pathlib import Path

# Data Path

In [13]:
data_folder = Path("../data")

## Load Dataset

In [14]:
sales_df = pl.read_parquet(data_folder / "sales_cleaned.parquet")

# Preview dataset
sales_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499


## Total Revenue and Transactions by City

In [15]:
city_performance = sales_df.group_by(["city"]).agg (
    pl.count().alias("transactions"),
    pl.col("revenue").sum().alias("total_revenue"),
    pl.col("price").mean().round().alias("average_order_value")
).sort("total_revenue", descending=True)

city_performance

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_21212\253708900.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


city,transactions,total_revenue,average_order_value
str,u32,i64,f64
"""South Dumdum""",446,144004,323.0
"""Berhampore""",421,138479,329.0
"""Allahabad""",435,137415,316.0
"""Ghaziabad""",406,133544,329.0
"""Haldia""",411,133489,325.0
…,…,…,…
"""Lucknow""",31,8619,278.0
"""Tumkur""",26,8224,316.0
"""Raiganj""",16,5634,352.0


## Festival Sales by City

In [16]:
festival_city = sales_df.group_by(["festival", "city"]).agg (
    pl.count().alias("transactions"),
    pl.col("revenue").sum().alias("total_revenue")
).sort(["festival", "total_revenue"], descending=[False, True])

festival_city.head(10)

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_21212\361083668.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


festival,city,transactions,total_revenue
str,str,u32,i64
null,"""South Dumdum""",355,114645
null,"""Allahabad""",354,111896
null,"""Meerut""",324,109526
null,"""Ghaziabad""",329,107821
null,"""Haldia""",326,105974
null,"""Berhampore""",322,105678
null,"""Nagercoil""",310,99590
null,"""Muzaffarpur""",287,93763
null,"""Uluberia""",284,93666


# City Revenue Share

In [19]:
city_performance = city_performance.with_columns(
   (
     pl.col("total_revenue")/pl.col("total_revenue").sum() * 100 
   ).round(1).alias("revenue_share_pct")
)

city_performance.sort("revenue_share_pct",descending = True)

city,transactions,total_revenue,average_order_value,revenue_share_pct
str,u32,i64,f64,f64
"""South Dumdum""",446,144004,323.0,0.9
"""Berhampore""",421,138479,329.0,0.9
"""Allahabad""",435,137415,316.0,0.8
"""Ghaziabad""",406,133544,329.0,0.8
"""Haldia""",411,133489,325.0,0.8
…,…,…,…,…
"""Lucknow""",31,8619,278.0,0.1
"""Tumkur""",26,8224,316.0,0.1
"""Raiganj""",16,5634,352.0,0.0


In [20]:
festival_city = sales_df.group_by(["festival", "city"]).agg(
    pl.count().alias("transactions"),
    pl.col("revenue").sum().alias("total_revenue")
).sort(["festival", "total_revenue"], descending=[False, True])

festival_city.head(10)

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_21212\1709528429.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


festival,city,transactions,total_revenue
str,str,u32,i64
null,"""South Dumdum""",355,114645
null,"""Allahabad""",354,111896
null,"""Meerut""",324,109526
null,"""Ghaziabad""",329,107821
null,"""Haldia""",326,105974
null,"""Berhampore""",322,105678
null,"""Nagercoil""",310,99590
null,"""Muzaffarpur""",287,93763
null,"""Uluberia""",284,93666


# Save Outputs

In [21]:
city_performance.write_parquet(data_folder / "city_performance.parquet")
festival_city.write_parquet(data_folder / "festival_city.parquet")